Preparing mock galaxy catalogue for *injections_with_SNR_for_sm.ipynb*.

The code was written by Mária Pálfi (marika97@student.elte.hu).

#### Read the data

In [1]:
import h5py
import pandas as pd
filename = "tao_maria.palfi_catalogue_4543/tao.4543.0.hdf5"

data = pd.DataFrame( data = [] )
with h5py.File(filename, "r") as f:
    for a in f.keys():
        print(a)
        # Get the data
        data[str(a)] = list(f[a])

Declination
Johnson_B_Absolute
Johnson_B_Apparent
Redshift_Cosmological
Redshift_Observed
Right_Ascension
Total_Stellar_Mass
TwoMASS_Ks_Absolute
TwoMASS_Ks_Apparent
WISE1_Absolute
WISE1_Apparent
WISE2_Absolute
WISE2_Apparent
X
Y
Z


#### Create numpy arrays

Convert AB system magnitudes to Vega system:

2MASS $K_s$:
$$m_\text{Vega} =  m_\text{AB} -1.85$$

Johnson $B$:
$$m_\text{Vega} =  m_\text{AB} + 0.09$$

WISE $W_1$:
$$ m_\text{Vega} =  m_\text{AB} -2.699$$

WISE $W_2$:
$$ m_\text{Vega} =  m_\text{AB} -3.339$$

Sources:

https://wise2.ipac.caltech.edu/docs/release/allsky/expsup/sec4_4h.html#conv2ab

https://iopscience.iop.org/article/10.1086/510127

In [2]:
import numpy as np

# the coordinates should be in radians
data.Declination = np.deg2rad( data.Declination )
data.Right_Ascension = np.deg2rad( data.Right_Ascension )

# create numpy arrays for the data
ra = np.array( data.Right_Ascension )
dec = np.array( data.Declination )
z = np.array( data.Redshift_Cosmological )
z_obs = np.array( data.Redshift_Observed )
stellar_mass = np.array( data.Total_Stellar_Mass )
m_B = np.array( data.Johnson_B_Apparent + 0.09 )
m_W1 = np.array( data.WISE1_Apparent - 2.699 )
m_W2 = np.array( data.WISE2_Apparent - 3.339)
m_Ks = np.array( data.TwoMASS_Ks_Apparent - 1.85 )

In [3]:
abs_B = np.array( data.Johnson_B_Absolute + 0.09 )
abs_W1 = np.array( data.WISE1_Absolute - 2.699 )
abs_W2 = np.array( data.WISE2_Absolute - 3.339 )
abs_Ks = np.array( data.TwoMASS_Ks_Absolute - 1.85 )

Minimums and maximums:

In [4]:
np.min(ra), np.max(ra), np.min(dec), np.max(dec), np.min(z)

(2.3230439944226e-07,
 6.283185225990617,
 -1.5705990178228162,
 1.570557082195359,
 0.0002044616006826548)

#### Create healpy indices, set the limits of the coordinates

In [5]:
import healpy as hp
nside = 1024
ind = hp.pixelfunc.ang2pix(nside,dec+np.pi/2,ra)

ra_dec_lim = 0
ra_min = 0.0
ra_max = np.pi*2.0
dec_min = -np.pi/2.0
dec_max = np.pi/2.0 

#### We assume that there is no error of the redshift

In [6]:
sigmaz = np.zeros( len(data) )

#### Write the data to a new hdf5 file

In [7]:
with h5py.File("mock_for_sm.hdf5", "w") as f:
    f.create_dataset("ra", data=ra)
    f.create_dataset("dec", data=dec)
    f.create_dataset("z", data=z)
    f.create_dataset("z_obs", data=z_obs)
    f.create_dataset("stellar_mass", data=stellar_mass)
    f.create_dataset("m_B", data=m_B)
    f.create_dataset("m_K", data=m_Ks)
    f.create_dataset("m_W1", data=m_W1)
    f.create_dataset("m_W2", data=m_W2)
    f.create_dataset("sigmaz", data=sigmaz)
    f.create_dataset("skymap_indices", data=ind)
    f.create_dataset("radec_lim", data=np.array([ra_dec_lim,ra_min,ra_max,dec_min,dec_max]))

In [8]:
from astropy.cosmology import FlatLambdaCDM
cosmo = FlatLambdaCDM(H0=73, Om0=0.25, Ob0=0.045)
dL = cosmo.luminosity_distance(z_obs)

In [9]:
dL.value

array([856.75966899, 867.71460097, 891.88261493, ...,  30.4931696 ,
        32.72633671,  20.95829784])

In [10]:
m_B = abs_B + 5*np.log10(dL.value) + 25
m_Ks = abs_Ks + 5*np.log10(dL.value) + 25
m_W1 = abs_W1 + 5*np.log10(dL.value) + 25
m_W2 = abs_W2 + 5*np.log10(dL.value) + 25

/tmp/ipykernel_25332/1403185730.py:1: RuntimeWarning: invalid value encountered in log10
  m_B = abs_B + 5*np.log10(dL.value) + 25
/tmp/ipykernel_25332/1403185730.py:2: RuntimeWarning: invalid value encountered in log10
  m_Ks = abs_Ks + 5*np.log10(dL.value) + 25
/tmp/ipykernel_25332/1403185730.py:3: RuntimeWarning: invalid value encountered in log10
  m_W1 = abs_W1 + 5*np.log10(dL.value) + 25
/tmp/ipykernel_25332/1403185730.py:4: RuntimeWarning: invalid value encountered in log10
  m_W2 = abs_W2 + 5*np.log10(dL.value) + 25


In [11]:
np.sum(dL < 0)

177

In [12]:
with h5py.File("mock_for_sm_abs.hdf5", "w") as f:
    f.create_dataset("ra", data=ra)
    f.create_dataset("dec", data=dec)
    f.create_dataset("z", data=z)
    f.create_dataset("z_obs", data=z_obs)
    f.create_dataset("stellar_mass", data=stellar_mass)
    f.create_dataset("m_B", data=m_B)
    f.create_dataset("m_K", data=m_Ks)
    f.create_dataset("m_W1", data=m_W1)
    f.create_dataset("m_W2", data=m_W1)
    f.create_dataset("sigmaz", data=sigmaz)
    f.create_dataset("skymap_indices", data=ind)
    f.create_dataset("radec_lim", data=np.array([ra_dec_lim,ra_min,ra_max,dec_min,dec_max]))

#### Check the file

In [1]:
import h5py
import pandas as pd
f = h5py.File('mock_for_sm_abs.hdf5', 'r')   

In [2]:
data = pd.DataFrame( data = [] )
for a in f.keys():
    if a == 'ra':
        print(a)
        # Get the data
        data[str(a)] = list(f[a])
    if a == 'dec':
        print(a)
        # Get the data
        data[str(a)] = list(f[a])
    if a == 'z':
        print(a)
        # Get the data
        data[str(a)] = list(f[a])
    if a == 'z_obs':
        print(a)
        # Get the data
        data[str(a)] = list(f[a])
    if a == 'm_K':
        print(a)
        # Get the data
        data[str(a)] = list(f[a])
    if a == 'stellar_mass':
        print(a)
        # Get the data
        data[str(a)] = list(f[a])

dec
m_K
ra
stellar_mass
z
z_obs


In [3]:
data.to_csv( 'mock_for_sm_minimal.txt' )

### Checking

In [1]:
import pandas as pd 
data = pd.read_csv('mock_for_sm_minimal.txt', index_col=0)

In [2]:
data.head()

,dec,m_K,ra,stellar_mass,z,z_obs
0,0.760462,20.178024,0.020387,0.053306,0.185029,0.182923
1,0.763832,17.183575,0.006789,0.618883,0.186787,0.185017
2,0.760068,16.939637,0.005451,0.843367,0.190252,0.189618
3,0.761422,14.663945,0.010789,8.170673,0.185415,0.183351
4,0.761146,16.740426,0.011293,0.811651,0.185504,0.182808
